# Early Pump Prediction for New Solana Memecoins

This notebook documents the full research workflow for predicting whether newly observed Solana tokens are likely to experience a large price increase after the initial observation period.

## 1. Abstract

This project investigates whether early token-level features can be used to predict large short-term price increases in newly observed Solana memecoins. A dataset of newly observed tokens was collected using the BirdEye API, with an initial snapshot taken near the observation time and a follow-up snapshot collected approximately 24 hours later. Tokens were labelled as positive examples if their follow-up USD price was at least 50% higher than their initial USD price.

A logistic regression classifier was trained using only features available from the initial snapshot, including liquidity, fully diluted valuation, holder count, token supply, transaction count, price, and a Whale Dominance Score feature. Model performance was evaluated using a train/test split, classification metrics, predicted probabilities, threshold testing, and error analysis.

The results suggest that early token data can be structured into a supervised learning problem, but the predictive reliability of this first baseline is limited by small sample size, class imbalance, missing holder data, and the high noise of newly launched memecoin markets. This notebook should therefore be interpreted as an initial research baseline rather than a production-ready trading model.

## 2. Introduction

I have observed that newly created Solana tokens seem to experience extreme price movements shortly after launch. The goal of this project is to investigate whether early token-level signals can be used to predict which tokens are more likely to experience a significant pump within a later observation window.

This project focuses on tokens that were newly observed at the time of sampling and uses only information available from the initial snapshot to make predictions.

## 3. Research Question

Can early token features from newly observed Solana tokens be used to predict whether a token will increase by at least 50% after the initial observation period?

### Target Definition

A token is labelled as a pump if its follow-up price is at least 50% higher than its initial price.

## 4. Data Collection Methodology

The dataset was collected using a separate set of Python scripts. The notebook does not directly call the API. Instead, it loads saved CSV files produced during the data collection process.

The collection process followed these steps:

1. Pull a candidate pool of newly observed Solana tokens.
2. Randomly sample a smaller tracking set from the candidate pool.
3. Collect an initial snapshot for each tracked token.
4. Collect a follow-up snapshot approximately 24 hours later.

### Data Files Used

- Candidate pool CSV
- Tracking set CSV
- Initial snapshot CSV
- Follow-up snapshot CSV

In [60]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import matplotlib.pyplot as plt

## 5. Load Data

This section loads the initial and follow-up token snapshots from saved CSV files.

In [61]:
DATA_DIR = Path("data/raw")

initial_path = DATA_DIR / "initial_tracking_snapshot_initial_2026-06-01_201522.csv"
followup_path = DATA_DIR / "followup_snapshot_followup_2026-06-02_201522.csv"

initial_df = pd.read_csv(initial_path)
followup_df = pd.read_csv(followup_path)

initial_df.head()
followup_df.head()
print(initial_df.shape)
print(followup_df.shape)

(200, 38)
(200, 19)


## 6. Basic Data Inspection

This section checks the structure of the loaded data, including column names, missing values, and duplicate token addresses.

In [62]:
initial_df.columns
followup_df.columns
initial_df.info()
followup_df.info()
initial_df.isna().sum().sort_values(ascending=False).head(20)
followup_df.isna().sum().sort_values(ascending=False).head(20)
initial_df["address"].duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 38 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   address                        200 non-null    str    
 1   name                           200 non-null    str    
 2   symbol                         200 non-null    str    
 3   initial_collected_at_unix      200 non-null    int64  
 4   initial_collected_at_utc       200 non-null    str    
 5   random_seed                    200 non-null    int64  
 6   sample_size                    200 non-null    int64  
 7   sampled_at_utc                 200 non-null    str    
 8   listing_time_unix              0 non-null      float64
 9   listing_time_utc               0 non-null      float64
 10  age_minutes_at_candidate_pull  0 non-null      float64
 11  circulating_supply             200 non-null    float64
 12  total_supply                   200 non-null    float64
 13  l

np.int64(0)

## 7. Merge Initial and Follow-Up Data

This section combines the initial and follow-up snapshots using the token address as the key.

In [63]:
df = initial_df.merge(
    followup_df,
    on="address",
    suffixes=("_initial", "_followup"),
    how="inner"
)

df.shape
df.head()

,address,name_initial,symbol_initial,initial_collected_at_unix,initial_collected_at_utc,random_seed,sample_size,sampled_at_utc,listing_time_unix,listing_time_utc,...,fdv_followup,market_cap_followup,holders_followup,volume_24h_usd_followup,trade_24h_count_followup,buy_24h_followup,sell_24h_followup,unique_wallet_24h_followup,price_change_24h_percent_birdeye,followup_error
0,8gaGBDZKgufByHo9UrcjZnPZrRito3amdPJSuEL9pump,GAH,GAH CHECK FUN,1780345772,2026-06-01T20:29:32+00:00,20260601,200,2026-06-01T20:15:22.024935+00:00,NaN,NaN,...,NaN,NaN,1,0.000000,NaN,0,0,0,NaN,NaN
1,4x9RvjCpa4NzwBbdZqxEHNUsVfQGdE9TSrGr4UDMpump,Perfect Gay Stocks,GAYMAN,1780345804,2026-06-01T20:30:04+00:00,20260601,200,2026-06-01T20:15:22.024935+00:00,NaN,NaN,...,NaN,NaN,2,0.000000,NaN,0,0,0,NaN,NaN
2,ELCeer4xYXpYXKrUzCS2riNDiWzhaFsrFMm2c6A3pump,Togi Gambles,TOGI,1780345836,2026-06-01T20:30:36+00:00,20260601,200,2026-06-01T20:15:22.024935+00:00,NaN,NaN,...,2384.372952,2384.372952,3,249.830316,NaN,1,2,2,4.990699,NaN
3,GzXEmN6WY9zGmPTobP5RSUdFBnc56h6NUJTE5gn3pump,Make X Twitter Again,Twitter,1780345869,2026-06-01T20:31:09+00:00,20260601,200,2026-06-01T20:15:22.024935+00:00,NaN,NaN,...,2276.999351,2276.999351,1,0.000000,NaN,0,0,0,0.000000,NaN
4,EM9mdn1wWJH3EtLke34xu2bbS8gYcd5iBQ2ZDkbVpump,100K views every 5mins!!!!,GayBatman,1780345902,2026-06-01T20:31:42+00:00,20260601,200,2026-06-01T20:15:22.024935+00:00,NaN,NaN,...,2292.575956,2292.575956,1,0.000000,NaN,0,0,0,0.000000,NaN


## 8. Label Construction

This section creates the target variable for supervised learning.

A token is labelled as a pump if its follow-up price is at least 50% greater than its initial price.

In [64]:
df["return_24h"] = (
    df["price_usd_followup"] - df["price_usd_initial"]
) / df["price_usd_initial"]

df["pumped_50"] = (df["return_24h"] >= 0.50).astype(int)

df[["address", "price_usd_initial", "price_usd_followup", "return_24h", "pumped_50"]].head()

df["pumped_50"].value_counts()
df["pumped_50"].value_counts(normalize=True)

pumped_50
0    0.99
1    0.01
Name: proportion, dtype: float64

## 9. Data Cleaning

This section removes rows that cannot be used for training, such as tokens with missing prices or invalid initial prices.


### Whale Dominance Score Handling

Some tokens may not have a valid Whale Dominance Score due to missing holder data or API limitations. These missing values are handled explicitly rather than silently dropped.

In [65]:
df = df[df["price_usd_initial"].notna()]
df = df[df["price_usd_followup"].notna()]
df = df[df["price_usd_initial"] > 0]

df.shape

# create a missingness indicator
df["wds_missing"] = df["wds"].isna().astype(int)
df["wds"] = df["wds"].fillna(0)

## 10. Feature Selection

This section selects the model input features.

Only initial snapshot features are used as model inputs. Follow-up columns are excluded because they represent future information and would cause data leakage.

In [67]:
feature_cols = [
    "liquidity",
    "fdv",
    "holders",
    "price_usd_initial",
    "price_sol_initial",
    "circulating_supply",
    "total_supply",
    "txns",
    "wds",
    "wds_missing",
]

X = df[feature_cols]
y = df["pumped_50"]

X.head()

,liquidity,fdv,holders,price_usd_initial,price_sol_initial,circulating_supply,total_supply,txns,wds,wds_missing
2,0.845513,2271.032545,4,0.000002,2.805057e-08,1.000000e+09,1.000000e+09,NaN,0.999190,0
3,0.136991,2276.999351,1,0.000002,2.812427e-08,1.000000e+09,1.000000e+09,NaN,0.000000,1
4,0.183406,2292.575956,1,0.000002,2.831725e-08,1.000000e+09,9.999795e+08,NaN,0.000000,1
5,16.971183,2294.711731,6,0.000002,2.834313e-08,9.999971e+08,9.999971e+08,NaN,0.982417,0
6,0.136910,2301.377978,1,0.000002,2.842539e-08,1.000000e+09,1.000000e+09,NaN,0.000000,1


## 11. Final Preprocessing

This section prepares the feature matrix for modelling by handling missing values and removing unusable rows.

In [68]:
model_df = df[feature_cols + ["pumped_50"]].copy()

model_df = model_df.replace([np.inf, -np.inf], np.nan)

print("Before cleaning:", model_df.shape)
print(model_df.isna().sum().sort_values(ascending=False))

for col in feature_cols:
    model_df[col] = model_df[col].fillna(0)

model_df = model_df.dropna(subset=["pumped_50"])

print("After cleaning:", model_df.shape)

X = model_df[feature_cols]
y = model_df["pumped_50"]

X.shape, y.shape

Before cleaning: (170, 11)
txns                  170
liquidity               0
fdv                     0
holders                 0
price_usd_initial       0
price_sol_initial       0
circulating_supply      0
total_supply            0
wds                     0
wds_missing             0
pumped_50               0
dtype: int64
After cleaning: (170, 11)


((170, 10), (170,))

## 12. Train/Test Split

This section splits the dataset into training and testing sets so that model performance can be evaluated on tokens the model did not train on.

In [69]:
class_counts = y.value_counts()

if class_counts.min() < 2:
    print("Not enough examples in both classes for stratified split.")
    print(class_counts)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=20260601
    )
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=20260601,
        stratify=y
    )

## 13. Cross-Validation

This section evaluates the logistic regression model using stratified cross-validation on the training set.

Cross-validation helps reduce dependence on one lucky or unlucky train/test split, which is especially important because the dataset is small.

In [70]:
log_reg_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("log_reg", LogisticRegression(max_iter=1000))
])

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

min_class_count = y_train.value_counts().min()

if min_class_count < 2:
    print("Not enough examples in both classes for cross-validation.")
else:
    n_splits = min(5, min_class_count)

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=20260601
    )

    cv_results = cross_validate(
        log_reg_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True
    )

    cv_summary = pd.DataFrame(cv_results).mean().sort_index()
    cv_summary

Not enough examples in both classes for cross-validation.


## 14. Final Logistic Regression Model

This section trains the final logistic regression model on the full training set after cross-validation.



In [71]:
model = log_reg_pipeline

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('log_reg', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not 

## 15. Model Evaluation

This section evaluates the model on the test set using classification metrics.

In [72]:
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

confusion_matrix(y_test, y_pred)

accuracy_score(y_test, y_pred)

precision_score(y_test, y_pred, zero_division=0)

recall_score(y_test, y_pred, zero_division=0)

f1_score(y_test, y_pred, zero_division=0)


              precision    recall  f1-score   support

           0       1.00      0.95      0.98        42
           1       0.33      1.00      0.50         1

    accuracy                           0.95        43
   macro avg       0.67      0.98      0.74        43
weighted avg       0.98      0.95      0.96        43



0.5

## 16. Prediction Probabilities

This section looks at the model's predicted probabilities instead of only its hard 0/1 classifications.

In [73]:
y_prob = model.predict_proba(X_test)[:, 1]

results_df = X_test.copy()
results_df["actual"] = y_test.values
results_df["predicted"] = y_pred
results_df["pump_probability"] = y_prob

results_df.sort_values("pump_probability", ascending=False).head(20)

,liquidity,fdv,holders,price_usd_initial,price_sol_initial,circulating_supply,total_supply,txns,wds,wds_missing,actual,predicted,pump_probability
78,4.527267e+04,25688.087714,2781,2.568411e-05,3.172363e-07,1.000000e+09,1.000000e+09,0.0,0.770561,0,1,1,1.000000
37,3.004685e+03,1120.781810,1512,1.120797e-06,1.384349e-08,9.999863e+08,9.999863e+08,0.0,0.747746,0,0,1,0.990402
181,2.223845e+04,35438.059604,1089,3.543806e-05,4.377119e-07,1.000000e+09,1.000000e+09,0.0,0.419316,0,0,1,0.803298
120,1.113748e+01,1.062737,916,1.062737e-09,1.312636e-11,1.000000e+09,1.000000e+09,0.0,0.199118,0,0,0,0.439727
133,8.073062e-02,38703.074492,28,3.870355e-05,4.780454e-07,9.999878e+08,9.999878e+08,0.0,0.998923,0,0,0,0.000729
160,8.561573e-08,5357.908142,11,2.892007e-05,3.572052e-07,1.852662e+08,1.852661e+08,0.0,0.960969,0,0,0,0.000666
86,8.462310e-08,2090.700703,11,1.390036e-05,1.716898e-07,1.504062e+08,1.504062e+08,0.0,0.887952,0,0,0,0.000660
97,4.130851e+01,2373.848615,8,2.410084e-06,2.976807e-08,9.849650e+08,9.849650e+08,0.0,0.993410,0,0,0,0.000622
5,1.697118e+01,2294.711731,6,2.294718e-06,2.834313e-08,9.999971e+08,9.999971e+08,0.0,0.982417,0,0,0,0.000610
122,1.365036e-01,2273.238753,2,2.273239e-06,2.807782e-08,1.000000e+09,1.000000e+09,0.0,1.000000,0,0,0,0.000592


## 17. Threshold Testing

This section tests different probability thresholds for classifying a token as a buy candidate.

In [74]:
threshold = 0.70

custom_pred = (y_prob >= threshold).astype(int)

print(classification_report(y_test, custom_pred, zero_division=0))

for threshold in [0.3, 0.5, 0.7, 0.9]:
    custom_pred = (y_prob >= threshold).astype(int)
    print(f"Threshold: {threshold}")
    print(classification_report(y_test, custom_pred, zero_division=0))
    print()

              precision    recall  f1-score   support

           0       1.00      0.95      0.98        42
           1       0.33      1.00      0.50         1

    accuracy                           0.95        43
   macro avg       0.67      0.98      0.74        43
weighted avg       0.98      0.95      0.96        43

Threshold: 0.3
              precision    recall  f1-score   support

           0       1.00      0.93      0.96        42
           1       0.25      1.00      0.40         1

    accuracy                           0.93        43
   macro avg       0.62      0.96      0.68        43
weighted avg       0.98      0.93      0.95        43


Threshold: 0.5
              precision    recall  f1-score   support

           0       1.00      0.95      0.98        42
           1       0.33      1.00      0.50         1

    accuracy                           0.95        43
   macro avg       0.67      0.98      0.74        43
weighted avg       0.98      0.95      0.96

## 18. Feature Interpretation

This section inspects the logistic regression coefficients to understand which features are associated with higher or lower predicted pump probability.

In [75]:
log_reg = model.named_steps["log_reg"]

coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": log_reg.coef_[0]
}).sort_values("coefficient", ascending=False)

coef_df

,feature,coefficient
2,holders,0.880031
0,liquidity,0.148689
8,wds,0.068089
7,txns,0.000000
4,price_sol_initial,-0.002708
3,price_usd_initial,-0.002708
1,fdv,-0.002730
6,total_supply,-0.027945
5,circulating_supply,-0.027949
9,wds_missing,-0.041059


## 19. Error Analysis

This section inspects examples where the model was correct and incorrect.

In [76]:
results_df["error_type"] = np.select(
    [
        (results_df["actual"] == 1) & (results_df["predicted"] == 1),
        (results_df["actual"] == 0) & (results_df["predicted"] == 0),
        (results_df["actual"] == 0) & (results_df["predicted"] == 1),
        (results_df["actual"] == 1) & (results_df["predicted"] == 0),
    ],
    [
        "true_positive",
        "true_negative",
        "false_positive",
        "false_negative",
    ],
    default="unknown"
)

results_df["error_type"].value_counts()

error_type
true_negative     40
false_positive     2
true_positive      1
Name: count, dtype: int64

In [77]:
results_df[results_df["error_type"] == "false_positive"].head()

,liquidity,fdv,holders,price_usd_initial,price_sol_initial,circulating_supply,total_supply,txns,wds,wds_missing,actual,predicted,pump_probability,error_type
37,3004.685469,1120.781810,1512,0.000001,1.384349e-08,9.999863e+08,9.999863e+08,0.0,0.747746,0,0,1,0.990402,false_positive
181,22238.454425,35438.059604,1089,0.000035,4.377119e-07,1.000000e+09,1.000000e+09,0.0,0.419316,0,0,1,0.803298,false_positive


In [78]:
results_df[results_df["error_type"] == "false_negative"].head()

,liquidity,fdv,holders,price_usd_initial,price_sol_initial,circulating_supply,total_supply,txns,wds,wds_missing,actual,predicted,pump_probability,error_type


## 20. Baseline Conclusions

This notebook successfully frames early Solana memecoin pump prediction as a supervised binary classification problem. The final dataset was built by merging initial and follow-up token snapshots, constructing a 24-hour return label, and training a logistic regression model using only information available at the initial observation time.

The main value of this baseline is methodological: it establishes a repeatable workflow for collecting token snapshots, creating labels, avoiding future-data leakage, training a model, evaluating predictions, and inspecting model errors. Logistic regression was chosen as the first model because it is simple, interpretable, and useful as a benchmark before testing more complex models.

The current results should be interpreted cautiously. Newly launched memecoins are extremely noisy, and the dataset is small relative to the difficulty of the prediction task. The 50% pump threshold may also create a highly imbalanced target, which can make metrics such as accuracy misleading. Precision, recall, F1 score, confusion matrix results, and predicted probabilities are therefore more informative than accuracy alone.

Overall, this first baseline does not prove that early memecoin pumps can be reliably predicted. However, it does show that the research pipeline is functional and can be extended with more data, improved features, alternative labels, tree-based models, and paper-trading evaluation.

## 21. Limitations

This section documents limitations of the current experiment.

Possible limitations:

- Small dataset size.
- High noise in new memecoin markets.
- Missing holder data for some tokens.
- API rate limits affecting data completeness.
- 24-hour return may not capture shorter pumps and dumps.
- A 50% threshold may be too strict or too simple.
- Logistic regression may be too simple for nonlinear behaviour.

## 22. Next Steps

Future improvements may include:

- Collecting more training rounds from the paper trading bot.
- Testing different pump thresholds.
- Testing regression instead of classification.
- Adding more robust liquidity and holder concentration features.
- Comparing logistic regression against tree-based models.
- Using prediction probabilities to rank tokens instead of only classifying them.
- Evaluating simulated paper trading performance.

## 23. References

- Xiang, Y., Rish, S. M. S., Fu, Q., Li, Y., Wang, Q., Yuen, T. H., & Yu, J. (2025). *Measuring Memecoin Fragility*. arXiv:2512.00377v1.

- BirdEye. (n.d.). *BirdEye API Documentation*. Used as the data source for Solana token market data and token-level API information.

- scikit-learn developers. (n.d.). *LogisticRegression*. scikit-learn documentation. Used for the baseline logistic regression classifier.

- scikit-learn developers. (n.d.). *Cross-validation and StratifiedKFold*. scikit-learn documentation. Used for train/test validation and stratified cross-validation.

- pandas development team. (n.d.). *pandas documentation*. Used for loading, merging, cleaning, and inspecting CSV datasets.

- NumPy developers. (n.d.). *NumPy documentation*. Used for numerical operations and error analysis logic.

## 24. Acknowledgements

This notebook was developed with assistance from OpenAI's ChatGPT for project planning, notebook structure, debugging, and explanation of machine learning workflow decisions. All code, modelling choices, data collection, interpretation, and final project decisions were reviewed and adapted by the author.